# Growth-Spurt — a quantitative teardown 🔬
### The long-short on real EDGAR data, look-ahead-free · a CI that swallows the headline · micro-caps, the investment factor, the survivorship direction

![Signal: None](https://img.shields.io/badge/Signal-None-c0392b?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Replicates on large caps?: Not supported](https://img.shields.io/badge/Replicates_on_large_caps%3F-Not_supported-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb). We rebuild the asset-growth long-short on SEC balance-sheet data for large caps — June-30(y+1) formation on 10-Ks already filed, July→June returns — and explain why the highest headline Sharpe on the list cannot be measured there, and why the panel's own bias makes the sign uninformative.

> ⚠️ **Not investment advice.** EDGAR `Assets` (10-K, `filed`-date enforced) + Yahoo monthly total returns, ~399 current S&P 500 members, fiscal 2009–2023 → return windows July 2010 → June 2025, as-of 2026-06-01, fp `8dec74717e92` ([`../docs/results.md`](../docs/results.md)). The universe is survivorship-biased and large-cap — both stated, and the bias direction is itself part of the analysis. Sources in [`docs/references.md`](../docs/references.md).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))  # repo root (quantlab/)
sys.path.insert(0, os.path.abspath(".."))        # study package (growth_spurt/)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.5); plt.rcParams["axes.grid"] = True
import numpy as np, pandas as pd
from growth_spurt import data, strategy as st
# Real panel, cache-first (built by examples/verify.py --fetch): EDGAR Assets + Yahoo
# July->June returns, formed June 30 of y+1 on 10-Ks already FILED by then (no look-ahead),
# prices pinned to the desk as-of; the in-progress window is excluded.
ag, fwd = data.fetch_panel()
h = st.quantile_hedge(ag, fwd, q=0.2)
mkt = st.market_annual(fwd).reindex(h.index)


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| Signal | **None** | hedge -3.4%/yr, SE 2.8%, *t* = -1.23, 95% CI [-9.3%, +2.5%] — a statistical zero on 15 years |
| Tradability | **Mirage** | 0.835 lives in micro-caps, not large caps |
| Replicates on large caps? | **Not supported** | not measurable on survivors — and the panel is biased *against* the effect, so the negative sign is uninformative |

> 💡 *In plain words:* the most attractive number on the list is the least tradable — and the one test a retail quant can easily run (current large caps) is, by construction, the one least able to find it.

## 1 · The claim, steelmanned

- **H₁:** low-asset-growth firms out-earn high-growth ones (positive hedge).
- **H₂ (the pitch):** the hedge delivers a Sharpe near the vendor's 0.835 on a tradable universe.
- **H₃:** the effect is a standalone anomaly, not a known factor.

## 2 · So what? — what rides on each

If H₂ holds, an annual balance-sheet sort beats the market handsomely. If it fails on large caps, the headline is a micro-cap artifact.

## 3 · How we'd know — the protocol

EDGAR `Assets` → fiscal-year asset growth → **form on June 30 of y+1, only from 10-Ks filed by then** (EDGAR's `filed` field; 10-Ks land Feb–Mar, late filers are dropped, not peeked at) → long bottom-20% / short top-20%, equal-weight → July(y+1)→June(y+2) returns, partial windows excluded → report the hedge **with SE, t and a Student-t 95% CI** (n is small; adjectives are not inference) → weigh the three standard explanations, including the *direction* of the survivorship bias.

## 4 · The teardown

### 4.1 The long-short, by formation year

In [2]:
display(h.round(3))
s=st.summary(h['hedge'])
print(f"hedge: mean {s['mean']:+.2%}/yr  SE {s['se']:.2%}  t {s['tstat']:+.2f}  "
      f"95% CI [{s['ci95_low']:+.1%}, {s['ci95_high']:+.1%}]  Sharpe {s['sharpe']:.2f}  "
      f"hit {s['hit_rate']:.0%}  n={s['n']}")

,low_growth,high_growth,hedge
2009,0.297,0.321,-0.025
2010,0.083,0.077,0.007
2011,0.314,0.334,-0.021
2012,0.308,0.335,-0.027
2013,0.086,0.243,-0.157
2014,0.055,0.032,0.023
2015,0.224,0.285,-0.061
2016,0.146,0.264,-0.118
2017,0.102,0.147,-0.045
2018,0.037,0.258,-0.220


hedge: mean -3.39%/yr  SE 2.75%  t -1.23  95% CI [-9.3%, +2.5%]  Sharpe -0.32  hit 33%  n=15


> 💡 *In plain words:* a slightly negative mean whose confidence interval comfortably contains zero — and contains a respectable positive premium too. On this panel **H₁ is not supported, and not refuted either**: fifteen yearly observations simply cannot tell -3.4% from 0% from +2%.

### 4.2 The two legs — and why their gap is uninformative here

In [3]:
print('long  (low growth) mean: %+.2f%%' % (st.summary(h['low_growth'])['mean']*100))
print('short (high growth) mean: %+.2f%%' % (st.summary(h['high_growth'])['mean']*100))
print('vendor headline Sharpe: 0.835  |  our hedge Sharpe: %.2f' % st.summary(h['hedge'])['sharpe'])

long  (low growth) mean: +18.83%
short (high growth) mean: +22.22%
vendor headline Sharpe: 0.835  |  our hedge Sharpe: -0.32


> 💡 *In plain words:* the fast growers edged out the slow ones here — but remember who is missing from a *current-member* panel: the high-growth firms that blew up and delisted. Those were the short leg's paydays. Censoring them inflates the surviving high-growth leg's mean, which mechanically pushes the hedge down. **H₂ finds no support** — the famous Sharpe is nowhere near — but the sign of our point estimate is the one thing this panel is *guaranteed* to get wrong if the effect is real.

### 4.3 Why the headline can't be found here — three standard reasons

1. **Micro-cap concentration.** Hou-Xue-Zhang (2020) show accounting anomalies, asset growth included, concentrate in the smallest names; a large-cap universe is the wrong place.
2. **The investment factor.** In Fama-French five-factor, asset growth *is* the CMA factor — already priced, not a free anomaly. **H₃ rejected** by the literature.
3. **Survivorship cuts against the strategy.** Current membership deletes delisted high-growth blow-ups — the short leg's best trades — so the surviving short leg looks artificially strong and the hedge artificially weak. A test biased *against* the effect failing to find it is **weak evidence**, and that is exactly why the third stamp reads `NOT SUPPORTED` rather than `BUSTED`.

## 5 · The verdict

H₂ unsupported on tradable names; H₁ unmeasurable here (CI [-9.3%, +2.5%] on a panel tilted against it); H₃ rejected by the factor literature → Signal `NONE` (no measurable premium on this panel), Tradability `MIRAGE`, replicates-on-large-caps `NOT SUPPORTED`.

## 6 · Could you trade it?

Only by shorting hundreds of micro-caps — illiquid, hard-to-borrow, survivorship- and data-fragile. The liquid version is a statistical zero. The 0.835 is a property of the untradable tail.

## 7 · Going further

Forks: (a) add a small/mid-cap universe to see the effect strengthen toward the micro-cap end; (b) residualise asset growth against the CMA factor — does anything remain? (c) **point-in-time membership** — the one fix that would make the sign informative, by restoring the short leg's delisted blow-ups. Backlog: [`docs/pwb_strategies_inventory.md`](../../../docs/pwb_strategies_inventory.md).